In [1]:
import logging
import os
import re
from types import SimpleNamespace
import ruamel.yaml

import numpy as np
import pandas as pd
import geopandas as gpd
import pypsa
import pytz
import ruamel.yaml
import xarray as xr
from helpers import (
    create_dummy_data,
    create_network_topology,
    cycling_shift,
    locate_bus,
    mock_snakemake,
    override_component_attrs,
    prepare_costs,
    safe_divide,
    three_2_two_digits_country,
    two_2_three_digits_country,
    lossy_bidirectional_links,
    three_2_two_digits_country,
)
from helpers_offgrid import(
    add_nice_carrier_names,
    calculate_annuity,
    _add_missing_carriers_from_costs,
    load_costs,
    create_import_profile,
    add_esc_shipping,
    add_shipping_meoh,
    add_shipping_lnh3,
    add_shipping_lh2,
    create_esc_network,
    )
from prepare_transport_data import prepare_transport_data
from add_export_supply_chain import (
    get_efficiency, 
    read_efficiencies,
    select_ports,
    get_shipping_distance,
    parse_json_to_geodataframe,
    )
# from add_export_supply_chain import *


/nfs/home/edd32710/.conda/envs/PES_Model/lib/python3.10/site-packages/pypsa/networkclustering.py:16: UserWarning: The namespace `pypsa.networkclustering` is deprecated and will be removed in PyPSA v0.24. Please use `pypsa.clustering.spatial instead`. 
  warnings.warn(


In [2]:
n0_path = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/results/TR_2050_shipping_lnh3_at_port_20240925h2Port/postnetworks/elec_s_345_ec_lv1.1_Co2L_3H_2050_0.091_AP_0export_shipping_lnh3.nc'

In [3]:
costs_path = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/data/costs_2050.csv'

In [4]:
h2export = 100 #TWh

### Load config 

In [5]:
file_path = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/config.yaml'

# Read the YAML file
yaml = ruamel.yaml.YAML()


with open(file_path, 'r') as file:
    yaml_content = yaml.load(file)

    techs = yaml_content["custom_data"]["renewables"]
    year = yaml_content["scenario"]["planning_horizons"][0]
    dr = yaml_content["costs"]["discountrate"][0]
    sopts = yaml_content["scenario"]["sopts"]
    demand_sc = yaml_content["scenario"]["demand"][0]
    country = yaml_content["countries"]


file_path_pypsa_earth = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/config.pypsa-earth.yaml'

with open(file_path_pypsa_earth, 'r') as file2:
    yaml_content2 = yaml.load(file2)



### Get the centroid of Turkey for buses location

In [6]:

# Load world shapefile data
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))

# Select a specific country by name, e.g., 'Germany'
country_TR = world[world['name'] == 'Turkey']

# Reproject the geometries to a projected CRS (e.g., EPSG:3857 - Web Mercator)
country_proj = country_TR.to_crs(epsg=4326)

# Get the centroid of the country in projected coordinates
centroid = country_proj.geometry.centroid

# Extract x and y coordinates of the centroid
x, y = centroid.x.values[0], centroid.y.values[0]



/tmp/ipykernel_2338781/546478652.py:2: FutureWarning: The geopandas.dataset module is deprecated and will be removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.
  world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
/tmp/ipykernel_2338781/546478652.py:11: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroid = country_proj.geometry.centroid


### Create elec network and assign RES generators

In [7]:
# Create an empty n
overrides = override_component_attrs("/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/data/override_component_attrs")
n = pypsa.Network(override_component_attrs=overrides)
n.name = "PyPSA-Earth"
n.set_snapshots(pd.date_range(freq=sopts[0], **yaml_content["snapshots"]))
n.snapshot_weightings[:] *= 8760.0 / n.snapshot_weightings.sum()
n.meta = yaml_content

Nyears = n.snapshot_weightings.objective.sum() / 8760.0

n.add("Carrier", "H2")
# n.add("Carrier", "H2")

# Add electricity bus
n.add(
    "Bus", 
    "TR_AC", 
    v_nom=380, 
    carrier="AC",
    country="TR",
    location="TR",
    x=x,
    y=y,
    )
    
nodes = n.buses[n.buses.carrier == "AC"].index
 


# Load reference network to get RES data from:
n0 = pypsa.Network(n0_path)

techs= ['solar', 'onwind', 'onwind2']

# Attach RES data into n
for tech in techs:
    print(tech)
    custom_res = n0.generators[n0.generators.carrier == tech]
    custom_res_index = n0.generators[n0.generators.carrier == tech].index
    custom_res_t = n0.generators_t.p_max_pu.filter(custom_res_index)
    profile= pd.DataFrame(custom_res_t.mean(axis=1), columns=["TR_AC"])
    print(profile.sum())

    n.madd(
        "Generator",
        nodes,
        " " + tech,
        bus=nodes,
        carrier=tech,
        p_nom_extendable=True,
        p_nom_max=custom_res["p_nom_max"].sum(),
        capital_cost=custom_res["capital_cost"].mean(),
        efficiency=1.0,
        p_max_pu=profile,
        lifetime=custom_res["lifetime"].iloc[0],
    )

/nfs/home/edd32710/.conda/envs/PES_Model/lib/python3.10/site-packages/pypsa/components.py:318: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attrs.loc[bool_b, "default"] = attrs.loc[bool_b].isin({True, "True"})
/nfs/home/edd32710/.conda/envs/PES_Model/lib/python3.10/site-packages/pypsa/components.py:318: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attrs.loc[bool_b, "default"] = attrs.loc[bool_b].isin({True, "True"})
/nfs/home/edd32710/.conda/envs/PES_Model/lib/python3.10/site-packages/pypsa/components.py:318: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value

solar
TR_AC    460.076015
dtype: float64
onwind
TR_AC    245.344812
dtype: float64
onwind2
TR_AC    57.249316
dtype: float64


In [8]:
# Prepare the costs dataframe
costs = prepare_costs(
    costs_path,
    yaml_content["costs"]["USD2013_to_EUR2013"],
    dr,
    Nyears,
    yaml_content["costs"]["lifetime"],
)

In [9]:
costs2 = load_costs(
        costs_path,
        yaml_content2["costs"],
        yaml_content2["electricity"],
        Nyears,
    )


In [10]:
elec_opts = yaml_content2["electricity"]
carriers = pd.Index(elec_opts["extendable_carriers"]["Generator"])
_add_missing_carriers_from_costs(n, costs, carriers)
add_nice_carrier_names(n, config=yaml_content2)

### Add Hydrogen

In [11]:
# Add Hydrogen bus
n.add(
    "Bus",
    "TR_AC H2",
    carrier="H2",
    x=x,
    y=y,
)


n.madd(
    "Link",
    nodes + " H2 Electrolysis",
    bus1=nodes + " H2",
    bus0=nodes,
    p_nom_extendable=True,
    carrier="H2 Electrolysis",
    efficiency=costs.at["electrolysis", "efficiency"],
    capital_cost=costs.at["electrolysis", "fixed"],
    lifetime=costs.at["electrolysis", "lifetime"],
)

n.madd(
    "Link",
    nodes + " H2 Fuel Cell",
    bus0=nodes + " H2",
    bus1=nodes,
    p_nom_extendable=True,
    carrier="H2 Fuel Cell",
    efficiency=costs.at["fuel cell", "efficiency"],
    # NB: fixed cost is per MWel
    capital_cost=costs.at["fuel cell", "fixed"]
    * costs.at["fuel cell", "efficiency"],
    lifetime=costs.at["fuel cell", "lifetime"],
)

h2_pot = n0.stores[n0.stores.carrier == "H2 UHS"].e_nom_max.sum()
h2_capital_cost = n0.stores[n0.stores.carrier == "H2 UHS"].capital_cost.mean()


n.madd(
    "Bus",
    nodes + " H2 UHS",
    location=nodes,
    carrier="H2 UHS",
    x=x,
    y=y,
)

n.madd(
    "Store",
    nodes + " H2 UHS",
    bus=nodes + " H2 UHS",
    e_nom_extendable=True,
    e_nom_max=h2_pot,
    e_cyclic=True,
    carrier="H2 UHS",
    capital_cost=h2_capital_cost,
    lifetime=costs.at["hydrogen storage underground", "lifetime"],
)

n.madd(
    "Link",
    nodes + " H2 UHS charger",
    bus0=nodes + " H2",
    bus1=nodes + " H2 UHS",
    carrier="H2 UHS charger",
    # efficiency=costs.at["battery inverter", "efficiency"] ** 0.5,
    # capital_cost=costs.at["battery inverter", "fixed"],
    p_nom_extendable=True,
    # lifetime=costs.at["battery inverter", "lifetime"],
)

n.madd(
    "Link",
    nodes + " H2 UHS discharger",
    bus0=nodes + " H2 UHS",
    bus1=nodes + " H2",
    carrier="H2 UHS discharger",
    efficiency=1,
    # capital_cost=costs.at["battery inverter", "fixed"],
    p_nom_extendable=True,
    # lifetime=costs.at["battery inverter", "lifetime"],
)

Index(['TR_AC H2 UHS discharger'], dtype='object', name='Bus')

### Attach Geothermal

In [12]:
# geothermal_pot = n0.generators[n0.generators.carrier == "geothermal"].p_nom_max.sum()

# n.madd(
#     "Generator",
#     nodes,
#     " " + "geothermal",
#     bus=nodes,
#     carrier="geothermal",
#     p_nom_extendable=True,
#     p_nom_max=geothermal_pot,
#     capital_cost=costs2.at["geothermal", "capital_cost"],
#     marginal_cost=costs2.at["geothermal", "marginal_cost"],
#     efficiency=costs2.at["geothermal", "efficiency"],
# )

### Attach Battery storage

In [13]:
n.add("Carrier", "battery")


n.madd(
    "Bus",
    nodes + " battery",
    location=nodes,
    carrier="battery",
    x=x,
    y=y,
)

n.madd(
    "Store",
    nodes + " battery",
    bus=nodes + " battery",
    e_cyclic=True,
    e_nom_extendable=True,
    carrier="battery",
    capital_cost=costs.at["battery storage", "fixed"],
    lifetime=costs.at["battery storage", "lifetime"],
)

n.madd(
    "Link",
    nodes + " battery charger",
    bus0=nodes,
    bus1=nodes + " battery",
    carrier="battery charger",
    efficiency=costs.at["battery inverter", "efficiency"] ** 0.5,
    capital_cost=costs.at["battery inverter", "fixed"] /2, # battery inverter represented by two links (charging and discharging), while costs in cost data are for bidirectional inverter --> correction here
    p_nom_extendable=True,
    lifetime=costs.at["battery inverter", "lifetime"],
    p_min_pu=0,
    p_max_pu=1,
)

n.madd(
    "Link",
    nodes + " battery discharger",
    bus0=nodes + " battery",
    bus1=nodes,
    carrier="battery discharger",
    efficiency=costs.at["battery inverter", "efficiency"] ** 0.5,
    marginal_cost=yaml_content["sector"]["marginal_cost_storage"] /2, # battery inverter represented by two links (charging and discharging), while costs in cost data are for bidirectional inverter --> correction here
    p_nom_extendable=True,
    lifetime=costs.at["battery inverter", "lifetime"],
    p_min_pu=0,
    p_max_pu=1,
)

Index(['TR_AC battery discharger'], dtype='object', name='Bus')

### Attach ESC

In [14]:
additional_costs_path = "/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/data/additional_costs_2050.csv"
additional_costs_input = prepare_costs(
    additional_costs_path,
    yaml_content["costs"]["USD2013_to_EUR2013"],
    dr,
    Nyears,
    yaml_content["costs"]["lifetime"],
)

costs = pd.concat([costs, additional_costs_input], ignore_index=False)

# Efficiencies
efficiencies_path = "/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/data/esc_data/efficiencies.csv"
efficiencies = read_efficiencies(
    efficiencies_path, yaml_content["scenario"]["planning_horizons"]
)


export_ports_path = "/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/data/export_ports.csv"
ports = pd.read_csv(
    export_ports_path,
    index_col=None,
    keep_default_na=False,
).squeeze()

ports = ports[ports.country.isin(country)]

exp_ports = ports.copy()

port_buses_n0 = {'Samsun':'TR.63_1_AC', 'Mersin':'TR.58_1_AC', 'Aliaga':'TR.41_1_AC'}

exp_ports.set_index("name", inplace=True)
# exp_ports["bus"] = 0
exp_ports["bus"] = exp_ports.index.map(port_buses_n0)

# Reset the index but keep 'name' as a column, and set 'bus' as the new index
exp_ports.reset_index(inplace=True)
exp_ports.set_index("bus", inplace=True)

# Add 3 export locations to attach ESC to them
n.madd(
    "Bus",
    exp_ports.index + " TR_AC",
    carrier="port_AC",
    x=exp_ports.x.values,
    y=exp_ports.y.values,
    location=exp_ports.index + " TR_AC",
    country='TR',
)

n.madd(
    "Link",
    exp_ports.index + " TR_AC" + " port",
    bus1=exp_ports.index + " TR_AC",
    bus0=nodes,
    p_nom_extendable=True,
    carrier="AC",
)
n.madd(
    "Bus",
    exp_ports.index + " TR_AC H2",
    carrier="port_H2",
    x=exp_ports.x.values,
    y=exp_ports.y.values,
    location=exp_ports.index + " TR_AC",
    country='TR',
)

n.madd(
    "Link",
    exp_ports.index + " TR_AC H2" + " port",
    bus1=exp_ports.index + " TR_AC H2",
    bus0=nodes + " H2",
    p_nom_extendable=True,
    carrier="H2",
)

exp_ports.loc[exp_ports.index,"bus"]=exp_ports.index + " TR_AC"
exp_ports.set_index("bus", inplace=True)

# get hydrogen export buses/ports
hydrogen_buses_ports = n.buses[n.buses.carrier == "port_H2"]

# List of ports electrictiy buses
exp_nodes = hydrogen_buses_ports.location.values

# List of all AC nodes
nodes_df = n.buses[n.buses.carrier == "port_AC"]
nodes = n.buses[n.buses.carrier == "port_AC"].index

import_ports_path = "/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/data/import_ports.csv"

# Import ports and nodes
import_ports = pd.read_csv(
    import_ports_path,
    index_col=None,
    keep_default_na=False,
)#.squeeze()

import_ports.set_index('name', inplace=True)
imp_nodes = import_ports.index

# Get the shipping Distances between the import and export ports
# get_shipping_distances(exp_ports, import_ports)


In [15]:
exp_ports.index + " H2" + " " + exp_ports.name

bus
TR.63_1_AC TR_AC    TR.63_1_AC TR_AC H2 Samsun
TR.58_1_AC TR_AC    TR.58_1_AC TR_AC H2 Mersin
TR.41_1_AC TR_AC    TR.41_1_AC TR_AC H2 Aliaga
dtype: object

In [16]:
n.buses[n.buses.carrier == "port_H2"]

,carrier,control,country,location,sub_network,type,unit,v_mag_pu_max,v_mag_pu_min,v_mag_pu_set,v_nom,x,y
Bus,,,,,,,,,,,,,
TR.63_1_AC TR_AC H2,port_H2,PQ,TR,TR.63_1_AC TR_AC,,,MWh,inf,0.0,1.0,1.0,36.34982,41.29961
TR.58_1_AC TR_AC H2,port_H2,PQ,TR,TR.58_1_AC TR_AC,,,MWh,inf,0.0,1.0,1.0,34.63333,36.79983
TR.41_1_AC TR_AC H2,port_H2,PQ,TR,TR.41_1_AC TR_AC,,,MWh,inf,0.0,1.0,1.0,26.93334,38.83316


In [17]:

# Export Supply Chain wildcard
export_esc = yaml_content["export"]["esc_scenarios"]["esc"][0]  

In [18]:
# Create import profile
import_profiles = create_import_profile(sopts, h2export, yaml_content)

INFO:helpers_offgrid:The yearly import demand is 100.0 TWh, profile generated based on esc_scenarios method and resampled to 3H


In [19]:
create_esc_network(n, nodes, exp_nodes, imp_nodes, exp_ports, export_esc, import_profiles, import_ports, nodes_df, yaml_content, efficiencies, costs)

/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/scripts/helpers_offgrid.py:1475: FutureWarning: The geopandas.dataset module is deprecated and will be removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.
  world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/scripts/helpers_offgrid.py:1480: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  import_country = world[world['iso_2'] == import_ports.country[0]]
/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/scripts/helpers_offgrid.py:1487: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a

/nfs/home/edd32710/.conda/envs/PES_Model/lib/python3.10/site-packages/pypsa/components.py:875: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  new_df.at[name, k] = typ(v)


In [20]:
add_esc_shipping(n, dr, efficiencies_path, yaml_content, export_esc)

INFO:helpers_offgrid:Increasing the round-trip travel time from 208h to 208h (+0.00%) to achieve more levelled supply by ship.
INFO:helpers_offgrid:Adding 4 shipping convoys to shipping route.
INFO:helpers_offgrid:Increasing the round-trip travel time from 214h to 224h (+4.67%) to achieve more levelled supply by ship.
INFO:helpers_offgrid:Adding 4 shipping convoys to shipping route.
INFO:helpers_offgrid:Increasing the round-trip travel time from 216h to 224h (+3.70%) to achieve more levelled supply by ship.
INFO:helpers_offgrid:Adding 4 shipping convoys to shipping route.
INFO:helpers_offgrid:Increasing the round-trip travel time from 140h to 146h (+4.29%) to achieve more levelled supply by ship.
INFO:helpers_offgrid:Adding 3 shipping convoys to shipping route.
INFO:helpers_offgrid:Increasing the round-trip travel time from 136h to 139h (+2.21%) to achieve more levelled supply by ship.
INFO:helpers_offgrid:Adding 2 shipping convoys to shipping route.
INFO:helpers_offgrid:Increasing the

PyPSA Network 'PyPSA-Earth'
Components:
 - Bus: 88
 - Carrier: 11
 - Generator: 3
 - Link: 117
 - Load: 1
 - Store: 36
Snapshots: 2920

In [21]:
n

PyPSA Network 'PyPSA-Earth'
Components:
 - Bus: 88
 - Carrier: 11
 - Generator: 3
 - Link: 117
 - Load: 1
 - Store: 36
Snapshots: 2920

### Solve

In [22]:
from helpers_offgrid import (
    prepare_network,
    solve_network,
)

from solve_network import (
    add_battery_constraints,
    add_nh3_store_cap,
)

from pypsa.linopf import ilopf, network_lopf
from pypsa.linopt import define_constraints, get_var, join_exprs, linexpr

from vresutils.benchmark import memory_logger

In [23]:
# tmpdir = yaml_content["solving"].get("tmpdir")

# if tmpdir is not None:
#     Path(tmpdir).mkdir(parents=True, exist_ok=True)
#     opts = yaml_content["scenario"]["opts"][0].split("-")
#     solve_opts = yaml_content["solving"]["options"]

# # fn ="/nimble/home/edd32710/projects/kikikiki/memory.log"
# # with memory_logger(filename=fn, interval=30.0) as mem:
# #     overrides = override_component_attrs("/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/data/override_component_attrs")
# #     n = pypsa.Network(n, override_component_attrs=overrides)



#     n = prepare_network(n, solve_opts)

#     n = solve_network(
#         n,
#         config=yaml_content,
#         yaml_content=yaml_content,
#         opts=yaml_content["scenario"]["opts"][0].split("-"),
        
#         # solver_dir=tmpdir,
#         # solver_logfile=snakemake.log.solver,
#     )
    
#     n.meta = dict(yaml_content)
#     n.export_to_netcdf("/nimble/home/edd32710/projects/kikikiki/trial1.nc")

# # # logging output to the terminal
# # print("Objective function: {}".format(n.objective))

# # # logging output to file
# # logger.info("Objective function: {}".format(n.objective))
# # logger.info("Objective constant: {}".format(n.objective_constant))
# # logger.info("Maximum memory usage: {}".format(mem.mem_usage))

In [24]:
# Path(tmpdir).mkdir(parents=True, exist_ok=True)
opts = yaml_content["scenario"]["opts"][0].split("-")
solve_opts = yaml_content["solving"]["options"]

# fn ="/nimble/home/edd32710/projects/kikikiki/memory.log"
# with memory_logger(filename=fn, interval=30.0) as mem:
#     overrides = override_component_attrs("/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/data/override_component_attrs")
#     n = pypsa.Network(n, override_component_attrs=overrides)



n = prepare_network(n, solve_opts)

n = solve_network(
    n,
    config=yaml_content,
    yaml_content=yaml_content,
    opts=yaml_content["scenario"]["opts"][0].split("-"),
    
    # solver_dir=tmpdir,
    # solver_logfile=snakemake.log.solver,
)

n.meta = dict(yaml_content)
n.export_to_netcdf("/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/results/TR_2050_shipping_lnh3_offcombined_grid_20240927/postnetworks/elec_s_348_ec_lv1.1_Co2L_3H_2050_0.091_AP_{}export_shipping_lnh3.nc".format(h2export))



INFO:pypsa.linopf:Prepare linear problem
INFO:pypsa.linopf:Total preparation time: 5.53s
INFO:pypsa.linopf:Solve linear problem using Gurobi solver


Set parameter TokenServer to value "10.186.19.42"
Read LP format model from file /tmp/pypsa-problem-9s63_6g8.lp
Reading time = 2.38 seconds
obj: 1273121 rows, 560797 columns, 2437590 nonzeros
Set parameter Threads to value 25
Set parameter Method to value 2
Set parameter Crossover to value 0
Set parameter BarConvTol to value 1e-06
Set parameter Seed to value 123
Set parameter AggFill to value 0
Set parameter PreDual to value 0
Set parameter GURO_PAR_BARDENSETHRESH to value 200
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: AMD EPYC 7542 32-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 32 physical cores, 64 logical processors, using up to 25 threads

Optimize a model with 1273121 rows, 560797 columns and 2437590 nonzeros
Model fingerprint: 0xa2785328
Coefficient statistics:
  Matrix range     [1e-03, 5e+00]
  Objective range  [3e-02, 2e+05]
  Bounds range     [1e+04, 1e+08]
  RHS range        [1e+04, 1e+04]
Presolve removed 

INFO:pypsa.linopf:Optimization successful. Objective value: 9.28e+09
INFO:pypsa.io:Exported network elec_s_348_ec_lv1.1_Co2L_3H_2050_0.091_AP_100export_shipping_lnh3.nc has links, carriers, buses, stores, generators, loads


<xarray.Dataset>
Dimensions:                       (snapshots: 2920, investment_periods: 0,
                                   links_i: 117, links_t_mu_lower_i: 100,
                                   links_t_mu_upper_i: 66, links_t_p0_i: 117,
                                   links_t_p1_i: 117, links_t_p2_i: 6,
                                   links_t_p3_i: 3, links_t_p_max_pu_i: 56,
                                   links_t_p_min_pu_i: 56, carriers_i: 11,
                                   buses_i: 88, buses_t_marginal_price_i: 88,
                                   buses_t_p_i: 57, stores_i: 36,
                                   stores_t_e_i: 36, stores_t_p_i: 36,
                                   generators_i: 3, generators_t_p_i: 3,
                                   generators_t_p_max_pu_i: 3, loads_i: 1,
                                   loads_t_p_i: 1, loads_t_p_set_i: 1)
Coordinates: (12/24)
  * snapshots                     (snapshots) int64 0 1 2 3 ... 2917 2918 2919
  * investment_periods            (investment_periods) int64 
  * links_i                       (links_i) object 'TR_AC H2 Electrolysis' .....
  * links_t_mu_lower_i            (links_t_mu_lower_i) object 'TR_AC H2 Elect...
  * links_t_mu_upper_i            (links_t_mu_upper_i) object 'TR_AC H2 Elect...
  * links_t_p0_i                  (links_t_p0_i) object 'TR_AC H2 Electrolysi...
    ...                            ...
  * generators_i                  (generators_i) object 'TR_AC solar' ... 'TR...
  * generators_t_p_i              (generators_t_p_i) object 'TR_AC solar' ......
  * generators_t_p_max_pu_i       (generators_t_p_max_pu_i) object 'TR_AC onw...
  * loads_i                       (loads_i) object 'NH3 export load'
  * loads_t_p_i                   (loads_t_p_i) object 'NH3 export load'
  * loads_t_p_set_i               (loads_t_p_set_i) object 'NH3 export load'
Data variables: (12/69)
    snapshots_snapshot            (snapshots) datetime64[ns] 2013-01-01 ... 2...
    snapshots_objective           (snapshots) float64 3.0 3.0 3.0 ... 3.0 3.0
    snapshots_stores              (snapshots) float64 3.0 3.0 3.0 ... 3.0 3.0
    snapshots_generators          (snapshots) float64 3.0 3.0 3.0 ... 3.0 3.0
    investment_periods_objective  (investment_periods) object 
    investment_periods_years      (investment_periods) object 
    ...                            ...
    generators_t_p                (snapshots, generators_t_p_i) float64 0.0 ....
    generators_t_p_max_pu         (snapshots, generators_t_p_max_pu_i) float64 ...
    loads_bus                     (loads_i) object 'NH3 export load bus'
    loads_carrier                 (loads_i) object 'NH3'
    loads_t_p                     (snapshots, loads_t_p_i) float64 1.142e+04 ...
    loads_t_p_set                 (snapshots, loads_t_p_set_i) float64 1.142e...
Attributes:
    network__cCounter:           1273122
    network__multi_invest:       0
    network__xCounter:           560798
    network_name:                PyPSA-Earth
    network_objective:           9283905554.289103
    network_objective_constant:  0.0
    network_pypsa_version:       0.24.0
    network_srid:                4326
    meta:                        {"logging_level": "INFO", "tutorial": false,...

In [241]:
yaml_content["export"]["esc_scenarios"]["synthesis"] == 'free'

False

In [242]:
'NH3 export load' in n.loads_t['p'].columns

True

In [243]:
n.links_t['p3'].filter(like='Haber-Bosch')

Link,Samsun TR_AC Haber-Bosch,Mersin TR_AC Haber-Bosch,Aliaga TR_AC Haber-Bosch
snapshot,,,
2013-01-01 00:00:00,1.820878e-21,1.711148e-21,1.738846e-21
2013-01-01 03:00:00,1.824613e-21,1.712312e-21,1.739851e-21
2013-01-01 06:00:00,1.827085e-21,1.716216e-21,1.742461e-21
2013-01-01 09:00:00,1.825540e-21,1.715174e-21,1.740587e-21
2013-01-01 12:00:00,1.824177e-21,1.713658e-21,1.739532e-21
...,...,...,...
2013-12-31 09:00:00,1.817308e-21,1.706470e-21,1.733048e-21
2013-12-31 12:00:00,1.824447e-21,1.711072e-21,1.737584e-21
2013-12-31 15:00:00,1.829506e-21,1.713324e-21,1.742462e-21


In [219]:
yaml_content["sector"]["ammonia"]

{'network_limit': 48612, 'storage_limit': 25000}

In [220]:
n.stores.loc[(n.stores.carrier == "NH3 store")].e_nom_opt.sum()/1e6

3.380205164409701e-17

In [221]:
n.generators.p_nom_opt

Generator
TR_AC solar      1.666124e-15
TR_AC onwind     4.303265e-16
TR_AC onwind2    7.098175e-17
Name: p_nom_opt, dtype: float64

In [222]:
n.generators.p_nom_max

Generator
TR_AC solar      2.136928e+06
TR_AC onwind     7.309888e+05
TR_AC onwind2    1.047195e+04
Name: p_nom_max, dtype: float64

In [223]:
n.generators_t.p_max_pu#.filter(like='solar', axis=1)

Generator,TR_AC onwind,TR_AC onwind2,TR_AC solar
snapshot,,,
2013-01-01 00:00:00,0.176952,0.051090,0.000000
2013-01-01 03:00:00,0.204936,0.063113,0.000000
2013-01-01 06:00:00,0.261937,0.080424,0.223226
2013-01-01 09:00:00,0.284096,0.073580,0.486887
2013-01-01 12:00:00,0.396777,0.116849,0.294301
...,...,...,...
2013-12-31 09:00:00,0.000000,0.000000,0.262488
2013-12-31 12:00:00,0.010651,0.000000,0.192906
2013-12-31 15:00:00,0.011920,0.000000,0.000000


In [224]:
n.links

,bus1,bus0,p_nom_extendable,carrier,efficiency,capital_cost,lifetime,build_year,bus2,bus3,...,ramp_limit_shut_down,ramp_limit_start_up,ramp_limit_up,shut_down_cost,start_up_cost,terrain_factor,type,up_time_before,scale_costs_based_on,charger_ratio
Link,,,,,,,,,,,,,,,,,,,,,
TR_AC H2 Electrolysis,TR_AC H2,TR_AC,True,H2 Electrolysis,0.750000,26159.892648,35.0,0.0,,,...,1.0,1.0,NaN,0.0,0.0,1.0,,1,NaN,NaN
TR_AC H2 Fuel Cell,TR_AC,TR_AC H2,True,H2 Fuel Cell,0.500000,82602.649562,10.0,0.0,,,...,1.0,1.0,NaN,0.0,0.0,1.0,,1,NaN,NaN
TR_AC H2 UHS charger,TR_AC H2 UHS,TR_AC H2,True,H2 UHS charger,1.000000,0.000000,inf,0.0,,,...,1.0,1.0,NaN,0.0,0.0,1.0,,1,NaN,NaN
TR_AC H2 UHS discharger,TR_AC H2,TR_AC H2 UHS,True,H2 UHS discharger,1.000000,0.000000,inf,0.0,,,...,1.0,1.0,NaN,0.0,0.0,1.0,,1,NaN,NaN
TR_AC battery charger,TR_AC battery,TR_AC,True,battery charger,0.979796,4965.198717,10.0,0.0,,,...,1.0,1.0,NaN,0.0,0.0,1.0,,1,NaN,-2482.633938
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
NH3 (l) transport ship convoy 2 - Aliaga TR_AC --> Damietta unloading,Damietta berth (imp),NH3 (l) transport ship convoy 2 - Aliaga TR_AC...,True,NH3,1.000000,0.000000,inf,0.0,,,...,1.0,1.0,NaN,0.0,0.0,1.0,,1,NaN,NaN
NH3 (l) transport ship convoy 2 - Aliaga TR_AC --> Damietta trip demand & losses,NH3 (l) transport ship convoy 2 - Aliaga TR_AC...,NH3 (l) transport ship convoy 2 - Aliaga TR_AC...,True,NH3,0.995921,0.000000,inf,0.0,,,...,1.0,1.0,NaN,0.0,0.0,1.0,,1,NaN,NaN
NH3 (l) transport ship convoy 3 - Aliaga TR_AC --> Damietta loading,NH3 (l) transport ship convoy 3 - Aliaga TR_AC...,Aliaga TR_AC berth (exp),True,NH3,1.000000,0.000000,inf,0.0,,,...,1.0,1.0,NaN,0.0,0.0,1.0,,1,NaN,NaN


In [225]:
n.objective

3.17191211512507e-09

In [226]:
n

PyPSA Network 'PyPSA-Earth'
Components:
 - Bus: 88
 - Carrier: 11
 - Generator: 3
 - Link: 117
 - Load: 1
 - Store: 36
 - SubNetwork: 88
Snapshots: 2920

In [227]:
from pes_analysis_helpers import*

In [228]:
calc_ptx_demand(n)

0.0

In [229]:
calc_anh3s_capa_exp(n, agg=True)

8.951430158372843e-17

In [230]:
calc_batt_capa(n)

6.110470001330325e-21

In [231]:
calc_uhs_capa(n, agg=True)

5.531519561497319e-19

In [232]:
'NH3 export load' in n.loads_t['p'].columns

True

In [233]:
if 'NH3 export load' in n.loads_t['p'].columns:
    d_h2 = n.links_t['p3'].filter(like='Haber-Bosch') #Amounts of H2 produced at all nodes
    d_h2.columns = n.links.loc[d_h2.columns, 'bus3']
elif 'H2 export load' in n.loads_t['p'].columns:
    d_h2 = n.links_t['p0'].filter(like='H2 liquefaction') #Amounts of H2 produced at all nodes
    d_h2.columns = n.links.loc[d_h2.columns, 'bus0']
elif 'meOH export load' in n.loads_t['p'].columns:
    d_h2 = n.links_t['p1'].filter(like='methanolisation') #Amounts of H2 produced at all nodes
    d_h2.columns = n.links.loc[d_h2.columns, 'bus1']      

conversion_fact=33.3 # MWh/t_H2

weightings = pd.DataFrame(
    np.outer(n.snapshot_weightings["generators"], [1.0] * len(d_h2.T)),
    index=n.snapshots,
    columns=d_h2.columns,
)
d_h2 = d_h2 * weightings
# d_h2.columns = n.links.loc[d_h2.columns, 'bus0']
d_h2 = d_h2.groupby(d_h2.columns, axis=1).sum()


marginal_prices = n.buses_t.marginal_price.loc[:, d_h2.columns] #Marginal H2 prices at export nodes
h2_costs = (d_h2 * marginal_prices).sum()

/tmp/ipykernel_3702965/2232106857.py:20: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  d_h2 = d_h2.groupby(d_h2.columns, axis=1).sum()


In [234]:
d_h2

bus3,Aliaga TR_AC H2,Mersin TR_AC H2,Samsun TR_AC H2
snapshot,,,
2013-01-01 00:00:00,5.967634e-21,5.872129e-21,6.266596e-21
2013-01-01 03:00:00,5.966458e-21,5.870601e-21,6.275705e-21
2013-01-01 06:00:00,5.970974e-21,5.879133e-21,6.280246e-21
2013-01-01 09:00:00,5.966824e-21,5.876554e-21,6.276960e-21
2013-01-01 12:00:00,5.966088e-21,5.872823e-21,6.274867e-21
...,...,...,...
2013-12-31 09:00:00,5.953029e-21,5.861861e-21,6.257920e-21
2013-12-31 12:00:00,5.966632e-21,5.874049e-21,6.278078e-21
2013-12-31 15:00:00,5.974846e-21,5.877424e-21,6.289721e-21
